# Publish Silver Layer

**Author:** Databricks Ingestion Team  
**Version:** 1.0  
**Last Updated:** 2026-03-31

This notebook publishes curated data to the Silver layer, applying business logic and data quality checks. Databricks-ready and production best practices applied.

---

**Parameters:**
- `input_table`: Name of the Bronze table to read from (set via widget)
- `output_table`: Name of the Silver table to write to

---

**Instructions:**
1. Set the input and output table names using the widgets below.
2. Run all cells to publish data to the Silver layer.

---

**Notebook Metadata:**
- **Tested on:** Databricks Runtime 12.2 LTS
- **Contact:** data-eng@databricks.com


In [ ]:
# Databricks widgets for parameterization
import pyspark.sql.functions as F
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("publish_silver")

dbutils.widgets.text("input_table", "bronze_table", "Input Bronze Table")
dbutils.widgets.text("output_table", "silver_table", "Output Silver Table")
input_table = dbutils.widgets.get("input_table")
output_table = dbutils.widgets.get("output_table")

try:
    # Read from Bronze table
    df = spark.table(input_table)
    logger.info(f"Loaded {df.count()} records from {input_table}")
    # Example transformation: filter and select columns
    df_silver = df.filter(F.col("status") == "valid").select("id", "name", "amount", "timestamp")
    # Data quality check
    assert df_silver.count() > 0, "No valid records to publish."
    # Write to Silver table
    df_silver.write.format("delta").mode("overwrite").saveAsTable(output_table)
    logger.info(f"Published {df_silver.count()} records to {output_table}")
    print("Publish to Silver layer successful.")
except Exception as e:
    logger.error(f"Error in Silver layer publishing: {e}")
    print(f"Error: {e}")

# Resource cleanup
if 'df' in locals():
    df.unpersist(blocking=True)
if 'df_silver' in locals():
    df_silver.unpersist(blocking=True)


---

## Validation & Testing

Below, we validate the notebook logic with a simple assertion to ensure data was published. Add more tests as needed for your data.


In [ ]:
# Simple validation
try:
    df_test = spark.table(output_table)
    assert df_test.count() > 0, "No records found in Silver table."
    print("Validation passed: Data published to Silver layer.")
except Exception as e:
    print(f"Validation failed: {e}")


---

## Resource Cleanup

Unpersist DataFrames and clean up resources if needed to avoid memory issues in production workflows.

In [ ]:
# Resource cleanup
if 'df_test' in locals():
    df_test.unpersist(blocking=True)
    print("Resources cleaned up.")
